# Tabelas Bronze e Silver - Women's Clothing Reviews + Rent the Runway + US Holiday Dates

Consulta das tabelas criadas pelos notebooks `2.bronze_womens_rtr.ipynb` e `3.silver_womens_rtr.ipynb`.

In [1]:
from pyspark.sql import SparkSession
from delta import *

builder = SparkSession \
    .builder \
    .appName("Ver tabelas Bronze e Silver") \
    .config("spark.sql.warehouse.dir", "hdfs://hdfs-nn:9000/warehouse") \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .enableHiveSupport()

spark = configure_spark_with_delta_pip(builder).getOrCreate()

## Camada Bronze (dados originais, tal como vêm da fonte)

In [2]:
bronze = ["bronze_reviews", "bronze_rtr_reviews", "bronze_us_holidays"]
for t in bronze:
    print(f"bronze.{t}: {spark.table(f'bronze.{t}').count()} linhas")

bronze.bronze_reviews: 23486 linhas
bronze.bronze_rtr_reviews: 192544 linhas
bronze.bronze_us_holidays: 342 linhas


In [3]:
spark.table("bronze.bronze_reviews").show(5, truncate=25)

+----------+-----------+---+-----------------------+-------------------------+------+---------------+-----------------------+--------------+---------------+----------+-------------------------+-------------------------+
|Unnamed: 0|Clothing ID|Age|                  Title|              Review Text|Rating|Recommended IND|Positive Feedback Count| Division Name|Department Name|Class Name|           _data_ingestao|         _ficheiro_origem|
+----------+-----------+---+-----------------------+-------------------------+------+---------------+-----------------------+--------------+---------------+----------+-------------------------+-------------------------+
|         0|        767| 33|                   null|Absolutely wonderful -...|     4|              1|                      0|     Initmates|       Intimate| Intimates|2026-11-11 18:09:02.74...|hdfs://hdfs-nn:9000/pr...|
|         1|       1080| 34|                   null|Love this dress!  it's...|     5|              1|                   

In [4]:
spark.table("bronze.bronze_rtr_reviews").show(5, truncate=20)

+---+---------+---------+--------+-----+------+-------+------+----------+-----------------+--------------------+--------------------+----+-------+------+--------------------+--------------------+
|age|body type|bust size|category|  fit|height|item_id|rating|rented for|      review_date|      review_summary|         review_text|size|user_id|weight|      _data_ingestao|    _ficheiro_origem|
+---+---------+---------+--------+-----+------+-------+------+----------+-----------------+--------------------+--------------------+----+-------+------+--------------------+--------------------+
| 40|hourglass|      34c|   shift|  fit| 5' 3"|1226293|    10|   wedding|    June 17, 2014|Not heavy and was...|The fit was great...|   1|  31551|116lbs|2026-11-11 18:09:...|hdfs://hdfs-nn:90...|
| 24|full bust|     null|    gown|large| 5' 3"| 132738|    10|   wedding| December 9, 2013|I didn't want to ...|OMG ! For starter...|   8|  15901|  null|2026-11-11 18:09:...|hdfs://hdfs-nn:90...|
| 38|   petite|     

In [5]:
spark.table("bronze.bronze_us_holidays").show(5, truncate=False)

+----------+-----------+---------+-----+---+----+--------------------------+--------------------------------------------------------------------------+
|Date      |Holiday    |WeekDay  |Month|Day|Year|_data_ingestao            |_ficheiro_origem                                                          |
+----------+-----------+---------+-----+---+----+--------------------------+--------------------------------------------------------------------------+
|2004-07-04|4th of July|Sunday   |7    |4  |2004|2026-11-11 18:09:19.947856|hdfs://hdfs-nn:9000/projeto/bronze/holidays/us_holiday_dates_2004_2021.csv|
|2005-07-04|4th of July|Monday   |7    |4  |2005|2026-11-11 18:09:19.947856|hdfs://hdfs-nn:9000/projeto/bronze/holidays/us_holiday_dates_2004_2021.csv|
|2006-07-04|4th of July|Tuesday  |7    |4  |2006|2026-11-11 18:09:19.947856|hdfs://hdfs-nn:9000/projeto/bronze/holidays/us_holiday_dates_2004_2021.csv|
|2007-07-04|4th of July|Wednesday|7    |4  |2007|2026-11-11 18:09:19.947856|hdfs://hdfs-

## Camada Silver (dados limpos e prontos para a Gold)

In [6]:
silver = ["avaliacoes", "produtos", "avaliacoes_rtr", "produtos_rtr", "feriados"]
for t in silver:
    print(f"silver.{t}: {spark.table(f'silver.{t}').count()} linhas")

silver.avaliacoes: 23465 linhas
silver.produtos: 1402 linhas
silver.avaliacoes_rtr: 43643 linhas
silver.produtos_rtr: 5475 linhas
silver.feriados: 12 linhas


In [7]:
spark.table("silver.avaliacoes").show(5)

+------------+-----------+--------------+------+---------+------------------+--------------------+
|id_avaliacao|clothing_id| division_name|rating|recomenda|avaliacao_negativa|avaliacao_incoerente|
+------------+-----------+--------------+------+---------+------------------+--------------------+
|           1|        767|     Intimates|     4|     true|             false|               false|
|           2|       1080|       General|     5|     true|             false|               false|
|           3|       1077|       General|     3|    false|             false|               false|
|           4|       1049|General Petite|     5|     true|             false|               false|
|           5|        847|       General|     5|     true|             false|               false|
+------------+-----------+--------------+------+---------+------------------+--------------------+
only showing top 5 rows



In [8]:
spark.table("silver.produtos").show(5)

+-----------+--------------+---------------+----------+
|clothing_id| division_name|department_name|class_name|
+-----------+--------------+---------------+----------+
|        767|     Intimates|       Intimate| Intimates|
|       1080|       General|        Dresses|   Dresses|
|       1077|       General|        Dresses|   Dresses|
|       1049|General Petite|        Bottoms|     Pants|
|        847|       General|           Tops|   Blouses|
+-----------+--------------+---------------+----------+
only showing top 5 rows



In [9]:
spark.table("silver.avaliacoes_rtr").show(5)

+----------------+-------+--------------+----+------+------------------+------+
|id_avaliacao_rtr|item_id|data_avaliacao|nota|nota_5|avaliacao_negativa|ajuste|
+----------------+-------+--------------+----+------+------------------+------+
|               1| 999837|    2017-06-01|  10|   5.0|             false|   fit|
|               2| 296430|    2017-06-01|  10|   5.0|             false|   fit|
|               3|2686655|    2017-06-01|  10|   5.0|             false|   fit|
|               4| 573593|    2017-06-01|  10|   5.0|             false|   fit|
|               5|1813420|    2017-06-01|   8|   4.0|             false| small|
+----------------+-------+--------------+----+------+------------------+------+
only showing top 5 rows



In [10]:
spark.table("silver.produtos_rtr").show(5)

+-------+---------+---------------+
|item_id|categoria|departamento_wc|
+-------+---------+---------------+
| 123373|     gown|        Dresses|
| 123793|     gown|        Dresses|
| 124204|    dress|        Dresses|
| 124553|    dress|        Dresses|
| 125424|    dress|        Dresses|
+-------+---------+---------------+
only showing top 5 rows



In [11]:
spark.table("silver.feriados").orderBy("data").show(truncate=False)

+----------+-----------------+
|data      |feriado          |
+----------+-----------------+
|2017-06-19|Juneteenth       |
|2017-07-04|4th of July      |
|2017-09-02|Labor Day Weekend|
|2017-09-03|Labor Day Weekend|
|2017-09-04|Labor Day        |
|2017-10-09|Columbus Day     |
|2017-11-11|Veterans Day     |
|2017-11-22|Thanksgiving Eve |
|2017-11-23|Thanksgiving Day |
|2017-12-24|Christmas Eve    |
|2017-12-25|Christmas Day    |
|2017-12-31|New Year's Eve   |
+----------+-----------------+



In [12]:
spark.stop()